# M15 — Segmentation, imbalance, Dice, and image-level splits

**Original guided lab · 75–100 minutes.** Read 20 min, predict/code 35 min, failure investigation 20 min, explain and transfer 15 min. Run all cells in order in a fresh kernel. All executed data are synthetic unless explicitly stated. No network, GPU, or external dataset is required.

Segmentation assigns labels or class probabilities to spatial samples. A binary lesion task predicts a lesion probability at each voxel; a multi-class anatomical task may require mutually exclusive labels, while some tumor-region tasks use overlapping label definitions. The output activation and loss must match that label structure. A sigmoid with independent channels and a softmax over exclusive classes are different contracts.

The unit of prediction is a pixel or voxel, but the unit of independent evaluation is usually an image or participant. Randomly splitting pixels from the same image can leak subject-specific intensity, acquisition, and anatomy. Training and test masks must therefore follow an image-level or participant-level split before feature extraction, patch sampling, or augmentation. Augmented copies stay with their source participant.

We create small noisy images with circular foreground masks. A pixel classifier receives intensity and a local mean as two features. This is a learned segmentation baseline, not a CNN. It establishes a useful reference before moving to a more complex architecture. The local mean provides neighborhood context while preserving spatial shape; it can also blur narrow structures. The pipeline learns feature scaling and logistic weights from training images only.

Foreground occupies a small fraction of the image. A background-only predictor can obtain high pixel accuracy while missing every lesion. Dice measures overlap relative to the predicted and true foreground sizes. Sensitivity and precision distinguish missed lesion pixels from false positives; boundary distances describe another aspect of quality. A single average Dice can hide individual failures, small-structure errors, and empty-mask cases. Specify the convention when both masks are empty and report the distribution across images.

Dice loss is often used during training, but an evaluation metric and an optimization objective are not interchangeable. A soft probability-based Dice objective has different behavior from thresholded evaluation Dice, especially for empty targets and batch aggregation. The offline example trains with logistic likelihood and evaluates thresholded overlap. It does not claim to implement a MONAI training recipe.

The failure experiment reverses the intensity relationship at test time while keeping shapes identical. A model trained to associate brightness with foreground can fail badly under a different contrast or preprocessing convention. A high score on synthetic circles cannot establish a robust MRI input contract. Ask AI to name modality, channel order, normalization, spatial spacing, target labels, and uncertainty handling before proposing a real model. Visual overlays remain necessary because a correct loss calculation does not guarantee correct spatial alignment.

## Transformation contract

Image batch and masks → image-wise train/test split → intensity/local-mean features → trained pixel probabilities → threshold masks → per-image Dice. Thresholding discards confidence; aggregate metrics discard failure location and participant heterogeneity.

## Ask your AI tutor

```text
Explain this notebook one transformation at a time.
Before each cell ask me to predict shapes, units, and a check.
Give edits in executable cells of at most 20 lines.
Keep the prescribed split, random seed, and tests intact.
Distinguish generated suggestions from executed results.
After the failure experiment, ask me to explain the mechanism.
```

In [1]:
import numpy as np
from scipy.ndimage import uniform_filter
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
rng=np.random.default_rng(415)
r,c=np.mgrid[:32,:32];images=[];masks=[]
for i in range(28):
    center=rng.integers(10,22,2);mask=(r-center[0])**2+(c-center[1])**2<rng.integers(4,7)**2
    masks.append(mask);images.append(mask.astype(float)+rng.normal(0,.18,(32,32)))
images=np.array(images);masks=np.array(masks)
features=np.stack([images,uniform_filter(images,size=(1,3,3))],axis=-1)
model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=300))
model.fit(features[:18].reshape(-1,2),masks[:18].ravel())
prob=model.predict_proba(features[18:].reshape(-1,2))[:,1].reshape(10,32,32)
assert prob.shape==masks[18:].shape


In [2]:
def dice(a,b):
    denom=a.sum()+b.sum()
    return 1. if denom==0 else 2*np.logical_and(a,b).sum()/denom
pred=prob>=.5
scores=np.array([dice(a,b) for a,b in zip(pred,masks[18:])])
background=np.zeros_like(pred)
print('Foreground fraction / background accuracy:',masks[18:].mean(),np.mean(background==masks[18:]))
print('Per-image Dice:',scores)
assert scores.mean()>.9 and dice(background,masks[18:])==0
bad_images=1-images[18:]
bad_features=np.stack([bad_images,uniform_filter(bad_images,size=(1,3,3))],axis=-1)
bad=model.predict(bad_features.reshape(-1,2)).reshape(10,32,32)
bad_scores=[dice(a,b) for a,b in zip(bad,masks[18:])]
print('Contrast-reversed mean Dice:',np.mean(bad_scores))
assert np.mean(bad_scores)<.2


Foreground fraction / background accuracy: 0.0619140625 0.9380859375
Per-image Dice: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Contrast-reversed mean Dice: 0.0


## Deliberate failure and repair

Reversing the intensity–label relationship defeats the learned baseline. The image dimensions still pass, so shape checks cannot establish modality compatibility. A second failure is reporting background accuracy as successful lesion detection. Repair the input contract and report foreground-sensitive metrics with image-level distributions and overlays.

## Your investigation

Plot the worst held-out image with its mask overlay. Explain why uniform filtering uses size `(1,3,3)` rather than smoothing across image identities. Describe how patch sampling and augmentation would preserve participant splits. Compare a tiny lesion with a large one shifted by the same number of pixels.

## Transfer to real neuroimaging

Move from this pixel baseline to MONAI only after specifying data and label contracts. Its BraTS example requires four aligned MRI contrasts and defined tumor regions. No MONAI model, pretrained weights, real MRI, or medical segmentation validation is executed here.

**Primary teaching sources, pinned where hosted on GitHub:**

- [NMA DL: image architectures](https://github.com/NeuromatchAcademy/course-content-dl/blob/caba36c513fb8139ac3c9e7503f7a769dadde25e/tutorials/W2D2_Convnets/student/W2D2_Tutorial1.ipynb)
- [MONAI BraTS model input/target documentation](https://huggingface.co/MONAI/brats_mri_segmentation/blob/main/docs/README.md)

Pinned upstream tutorials are a separate assignment; they have **not been executed** by this core lab. They may require data downloads, specialist dependencies, unfinished student cells, and additional compute.

## Exit questions and answer key

1. Why split images before pixels? **Pixels from one participant share information and are not independent held-out people.**
2. Can 95% accuracy miss every lesion? **Yes, if 95% of pixels are background.**